# Lab 12: Generative Models — LDA, QDA, Naive Bayes & KNN
> Week 12 | CLO3 | ISLP Ch.4.4–4.5

## บทนำสัปดาห์

สัปดาห์นี้เราจะขยายขอบเขต Classification ออกจาก Logistic Regression ไปสู่ **Generative Models** ซึ่งได้แก่ LDA (Linear Discriminant Analysis), QDA (Quadratic Discriminant Analysis) และ Naive Bayes แนวคิดสำคัญคือแทนที่จะ model Pr(Y|X) โดยตรง (discriminative) เราจะ model Pr(X|Y) และ Pr(Y) แยกกัน แล้วใช้ Bayes Theorem คำนวณ Pr(Y|X) กลับมา **เป้าหมายของ Lab นี้** คือให้นักศึกษาสามารถ fit LDA, QDA, GaussianNB และ KNN ได้ อธิบาย assumption ของแต่ละ method เปรียบเทียบ performance ผ่าน confusion matrix และ classification metrics และเลือก classifier ที่เหมาะสมกับ dataset ที่กำหนดได้ ใน Data Science จริง การเลือก classifier ที่ถูกต้องส่งผลโดยตรงต่อ accuracy ของระบบ — เช่น LDA ทำงานดีกับ medical data ที่มีขนาดเล็ก ขณะที่ KNN เหมาะกับ boundary ที่ซับซ้อน Lab นี้ใช้ Default dataset และ Stock Market (Smarket) dataset จาก ISLP เพื่อให้เห็น use case จริงในการ predict credit default และทิศทางของตลาดหุ้น

## สิ่งที่จะเรียนรู้
- LDA: discriminant function, linear boundary, assumption Gaussian shared Σ
- QDA: quadratic boundary, per-class Σ, LDA vs QDA trade-off
- Naive Bayes: independence assumption, Gaussian NB
- KNN Classifier: K effect on bias-variance trade-off
- การเปรียบเทียบ 5 classifiers ด้วย confusion matrix, ROC curve

In [ ]:
# ─── Import libraries ──────────────────────────────────────────────────────────
# วัตถุประสงค์: โหลด library ทั้งหมดที่จำเป็นสำหรับ Classification methods
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ─── Classification models ─────────────────────────────────────────────────────
# วัตถุประสงค์: import classifiers ทุกตัวที่จะเปรียบเทียบในสัปดาห์นี้
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

# ─── Evaluation tools ──────────────────────────────────────────────────────────
# วัตถุประสงค์: import เครื่องมือวัด performance ของ classifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report,
    ConfusionMatrixDisplay, accuracy_score,
    precision_score, recall_score, f1_score,
    roc_curve, roc_auc_score
)
from scipy.special import expit  # sigmoid function

# ─── Global settings ───────────────────────────────────────────────────────────
# วัตถุประสงค์: กำหนด random seed และขนาดกราฟ default สำหรับ reproducibility
np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')

print('Setup complete ✓')

## Part 1: โหลดข้อมูลและ Baseline Model

**Part นี้เราจะโหลด Default dataset และ Smarket dataset** เพื่อทำความเข้าใจโครงสร้างข้อมูลก่อน fit classifier ใดๆ Default dataset จะใช้ตลอด Lab นี้เป็น dataset หลัก ส่วน Smarket dataset เป็น dataset เสริมที่แสดงการ predict ทิศทางตลาดหุ้น S&P500 ซึ่งเป็น use case ที่น่าสนใจในโลก Finance ขั้นตอนที่จะทำ: (1) โหลดข้อมูล (2) EDA เบื้องต้น (3) เตรียม X, y สำหรับ classification

In [ ]:
# ─── โหลด Default dataset ──────────────────────────────────────────────────────
# วัตถุประสงค์: ใช้ข้อมูล credit card default สำหรับ binary classification
try:
    default = pd.read_csv('https://www.statlearning.com/s/Default.csv', index_col=0)
    default.columns = default.columns.str.lower()
    print('โหลด Default.csv สำเร็จ ✓')
except:
    # สร้างข้อมูล synthetic ที่มีลักษณะเดียวกับ Default dataset
    np.random.seed(0)
    n = 10000
    balance = np.clip(np.random.exponential(900, n), 0, 3000)
    income  = np.random.normal(35000, 15000, n)
    student = np.random.choice([0, 1], n, p=[0.7, 0.3])
    log_odds = -10.65 + 0.0055*balance - 0.000002*income - 0.65*student
    default_y = np.random.binomial(1, expit(log_odds))
    default = pd.DataFrame({
        'default': ['Yes' if d else 'No' for d in default_y],
        'student': ['Yes' if s else 'No' for s in student],
        'balance': balance,
        'income': income
    })
    print('ใช้ synthetic Default dataset ✓')

# ─── แปลง categorical เป็น numeric ────────────────────────────────────────────
# วัตถุประสงค์: sklearn ต้องการ numeric labels สำหรับ classification
default['default_num'] = (default['default'] == 'Yes').astype(int)
default['student_num'] = (default['student'] == 'Yes').astype(int)

print(f'Default dataset: {default.shape}')
print(f'Class balance: {default["default"].value_counts().to_dict()}')
default.head()

In [ ]:
# ─── โหลด Smarket dataset ──────────────────────────────────────────────────────
# วัตถุประสงค์: Smarket = S&P500 Stock Market data ปี 2001–2005
# ใช้ Lag1–Lag5 (% return วันก่อนหน้า) predict Direction (Up/Down)
try:
    smarket = pd.read_csv('https://www.statlearning.com/s/Smarket.csv', index_col=0)
    smarket.columns = smarket.columns.str.lower()
    print('โหลด Smarket.csv สำเร็จ ✓')
except:
    # synthetic Smarket dataset
    np.random.seed(1)
    n = 1250
    lags = np.random.normal(0, 1, (n, 5))
    volume = np.random.uniform(1, 2, n)
    today = np.random.normal(0, 1, n)
    direction = np.where(today > 0, 'Up', 'Down')
    smarket = pd.DataFrame(lags, columns=[f'lag{i+1}' for i in range(5)])
    smarket['volume'] = volume
    smarket['today'] = today
    smarket['direction'] = direction
    smarket['year'] = np.repeat(range(2001, 2006), 250)
    print('ใช้ synthetic Smarket dataset ✓')

# แปลง Direction เป็น binary
smarket['direction_num'] = (smarket['direction'] == 'Up').astype(int)

print(f'Smarket dataset: {smarket.shape}')
print(f'Class balance: {smarket["direction"].value_counts().to_dict()}')
smarket.head()

### 🔰 TODO 1 (Easy): EDA + Logistic Regression Baseline

เราต้องการเข้าใจข้อมูลก่อนเลือก classifier ที่เหมาะสม การดู distribution ของแต่ละ class และ correlation ระหว่าง features ช่วยให้เราตัดสินใจได้ว่า LDA (ที่ assume Gaussian) จะเหมาะสมหรือไม่ ในขั้นตอนนี้ให้คุณทำ EDA บน Default dataset และ fit Logistic Regression เป็น baseline เพื่อเปรียบเทียบกับ LDA/QDA ในขั้นตอนต่อไป

**สิ่งที่ต้องทำ:**
1. Plot boxplot ของ `balance` แยกตาม `default` (Yes vs No)
2. Split Default dataset: features = [balance, income, student_num], test_size=0.2
3. Fit Logistic Regression และแสดง accuracy, precision, recall, F1
4. Print confusion matrix

**ผลลัพธ์ที่คาดหวัง**: Logistic Regression accuracy ≈ 0.97, แต่ recall (default=Yes) ต่ำ เพราะ imbalanced data

In [ ]:
# TODO 1: EDA + Logistic Regression Baseline
# ─── TODO: เขียน code ที่นี่ ──────────────────────────────────────────────────

# 1. Boxplot balance vs default


# 2. Split data
X = default[['balance', 'income', 'student_num']]
y = default['default_num']
# X_train, X_test, y_train, y_test = ...


# 3. Fit Logistic Regression


# 4. Metrics + confusion matrix


## Part 2: Linear Discriminant Analysis (LDA)

**Part นี้เราจะ fit LDA บน Default dataset** เพื่อเรียนรู้วิธีที่ LDA ทำงาน LDA assume ว่า X|Y=k ~ N(μₖ, Σ) โดย Σ เป็น covariance matrix เดียวกันทุก class ทำให้ decision boundary เป็น **linear** ข้อดีคือ stable เมื่อ n เล็กหรือ classes well-separated เราจะเปรียบเทียบ LDA กับ Logistic Regression จาก TODO 1 และดูว่า performance ต่างกันหรือไม่

In [ ]:
# ─── Fit LDA บน Default dataset ───────────────────────────────────────────────
# วัตถุประสงค์: LDA ใช้ discriminant function δₖ(x) = xᵀΣ⁻¹μₖ - ½μₖᵀΣ⁻¹μₖ + log(πₖ)
# sklearn จัดการ estimation Σ̂, μ̂ₖ, π̂ₖ ให้อัตโนมัติ

# เตรียม data (ใช้จาก TODO 1)
X = default[['balance', 'income', 'student_num']]
y = default['default_num']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# สร้างและ fit LDA model
lda = LinearDiscriminantAnalysis()
lda.fit(X_train, y_train)

# แสดง class priors (π̂ₖ = nₖ/n)
print('Class priors (π̂ₖ):')
for k, prior in zip(lda.classes_, lda.priors_):
    print(f'  Class {k}: {prior:.4f}')

# แสดง class means (μ̂ₖ)
print('\nClass means (μ̂ₖ) for [balance, income, student_num]:')
for k, mean in zip(lda.classes_, lda.means_):
    print(f'  Class {k}: {mean.round(2)}')

In [ ]:
# ─── LDA Predictions + Evaluation ─────────────────────────────────────────────
# วัตถุประสงค์: ประเมิน LDA ด้วย metrics เดียวกับ Logistic Regression เพื่อเปรียบเทียบ

y_pred_lda = lda.predict(X_test)
y_proba_lda = lda.predict_proba(X_test)[:, 1]  # probability ของ class 1

print('LDA Results:')
print(f'  Accuracy  : {accuracy_score(y_test, y_pred_lda):.4f}')
print(f'  Precision : {precision_score(y_test, y_pred_lda, zero_division=0):.4f}')
print(f'  Recall    : {recall_score(y_test, y_pred_lda, zero_division=0):.4f}')
print(f'  F1        : {f1_score(y_test, y_pred_lda, zero_division=0):.4f}')

# Confusion matrix visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# LDA confusion matrix
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_lda,
    display_labels=['No Default', 'Default'],
    cmap='Blues', ax=axes[0])
axes[0].set_title('LDA — Confusion Matrix')

# Posterior probability distribution
axes[1].hist(y_proba_lda[y_test == 0], bins=40, alpha=0.6, label='No Default', color='steelblue')
axes[1].hist(y_proba_lda[y_test == 1], bins=40, alpha=0.6, label='Default', color='tomato')
axes[1].axvline(0.5, color='black', linestyle='--', label='Threshold 0.5')
axes[1].set_xlabel('P(Default | X) from LDA')
axes[1].set_ylabel('Count')
axes[1].set_title('LDA Posterior Probability Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

### 🔰 TODO 2 (Medium): QDA + LDA vs QDA Comparison

QDA เป็นการ relax assumption ของ LDA โดยให้แต่ละ class มี covariance matrix Σₖ ของตัวเอง ทำให้ decision boundary กลายเป็น **quadratic** QDA มี flexibility มากกว่า แต่ต้องการ parameters มากกว่า ทำให้ variance สูงกว่า LDA ในข้อนี้คุณจะ fit QDA เปรียบเทียบกับ LDA บน Default dataset และสังเกตว่า trade-off เป็นอย่างไร

**สิ่งที่ต้องทำ:**
1. Fit `QuadraticDiscriminantAnalysis` บน X_train, y_train (ใช้ตัวแปรเดิม)
2. Print confusion matrix ของ QDA
3. สร้าง comparison table แบบนี้:

| Method | Accuracy | Precision | Recall | F1 |
|--------|----------|-----------|--------|----|
| LDA    | ...      | ...       | ...    | ...|
| QDA    | ...      | ...       | ...    | ...|

4. อธิบาย: LDA หรือ QDA ดีกว่าสำหรับ Default dataset? เพราะอะไร? (1–2 ประโยค)

**Hint**: ใช้ `qda.predict()` และ `qda.predict_proba()` เช่นเดียวกับ LDA

In [ ]:
# TODO 2: QDA + LDA vs QDA Comparison
# ─── TODO: เขียน code ที่นี่ ──────────────────────────────────────────────────

# 1. Fit QDA


# 2. Predictions


# 3. Comparison table


# 4. คำอธิบาย:
# (เขียนในรูปแบบ comment หรือ print statement)


## Part 3: Naive Bayes + KNN

**Part นี้เราจะ fit Gaussian Naive Bayes และ KNN Classifier** Naive Bayes เป็น Generative model ที่ assume ว่า features เป็น independent กันภายใน class (Naive assumption) แม้ว่า assumption นี้ไม่ค่อยเป็นจริงในข้อมูลจริง แต่มักให้ผลดีในทางปฏิบัติ ส่วน KNN เป็น non-parametric method ที่ไม่ assume distribution ใดๆ แต่ใช้ K neighbors ที่ใกล้ที่สุดในการตัดสิน class การเปลี่ยน K เป็นการควบคุม bias-variance trade-off

In [ ]:
# ─── Gaussian Naive Bayes ──────────────────────────────────────────────────────
# วัตถุประสงค์: GaussianNB assume ว่า P(Xⱼ|Y=k) ~ N(μₖⱼ, σ²ₖⱼ)
# แต่ละ feature มี distribution เป็น Gaussian อิสระกันใน class k

gnb = GaussianNB()
gnb.fit(X_train, y_train)
y_pred_gnb = gnb.predict(X_test)

print('Gaussian Naive Bayes Results:')
print(f'  Accuracy  : {accuracy_score(y_test, y_pred_gnb):.4f}')
print(f'  Precision : {precision_score(y_test, y_pred_gnb, zero_division=0):.4f}')
print(f'  Recall    : {recall_score(y_test, y_pred_gnb, zero_division=0):.4f}')
print(f'  F1        : {f1_score(y_test, y_pred_gnb, zero_division=0):.4f}')

# แสดง per-class parameters (μₖⱼ, σₖⱼ)
print('\nNB Class Means (μₖⱼ):')
for k, mean in zip(gnb.classes_, gnb.theta_):
    print(f'  Class {k}: {mean.round(2)}')

In [ ]:
# ─── KNN Classifier ────────────────────────────────────────────────────────────
# วัตถุประสงค์: ทดสอบ K=1,3,5,10 เพื่อดู effect ของ K ต่อ performance
# K น้อย = overfitting (high variance), K มาก = underfitting (high bias)

k_values = [1, 3, 5, 10, 20]
knn_results = []

for k in k_values:
    # Fit KNN ด้วย K neighbors
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    y_pred_k = knn.predict(X_test)
    
    knn_results.append({
        'K': k,
        'Accuracy': round(accuracy_score(y_test, y_pred_k), 4),
        'Precision': round(precision_score(y_test, y_pred_k, zero_division=0), 4),
        'Recall': round(recall_score(y_test, y_pred_k, zero_division=0), 4),
        'F1': round(f1_score(y_test, y_pred_k, zero_division=0), 4)
    })

knn_df = pd.DataFrame(knn_results)
print('KNN Performance vs K:')
print(knn_df.to_string(index=False))

# Plot F1 vs K
plt.figure(figsize=(8, 4))
plt.plot(knn_df['K'], knn_df['F1'], 'o-', color='purple', lw=2, markersize=8)
plt.xlabel('K (number of neighbors)')
plt.ylabel('F1 Score')
plt.title('KNN: F1 Score vs K on Default Dataset')
plt.xticks(k_values)
plt.grid(True)
plt.tight_layout()
plt.show()

### 🔰 TODO 3 (Medium): Naive Bayes บน Smarket Dataset

เราได้ทดสอบ methods ต่างๆ บน Default dataset แล้ว ตอนนี้จะทดสอบบน Smarket dataset ซึ่งเป็น **Stock Market data** — งานคือ predict ว่าตลาดจะขึ้น (Up) หรือลง (Down) โดยใช้ Lag1–Lag5 เป็น features ข้อมูลนี้น่าสนใจเพราะ Logistic Regression บน Smarket ให้ผลไม่ดีนัก (ตลาดหุ้นมีความเป็น random walk สูง) เราจะดูว่า Naive Bayes และ LDA ทำได้ดีกว่าหรือไม่

**สิ่งที่ต้องทำ:**
1. ใช้ Smarket data: features = [lag1, lag2, lag3, lag4, lag5, volume], target = direction_num
2. Split: train = year ≤ 2004, test = year = 2005 (ไม่ใช้ random split เพราะเป็น time series)
3. Fit Logistic Regression, LDA, GaussianNB บน train set
4. เปรียบเทียบ accuracy ของทั้ง 3 method บน test set
5. สรุป: method ไหนดีที่สุดบน Smarket? เพราะอะไร?

**Hint**: ใช้ `smarket['year'] <= 2004` เป็น train mask

In [ ]:
# TODO 3: Naive Bayes บน Smarket Dataset
# ─── TODO: เขียน code ที่นี่ ──────────────────────────────────────────────────

# 1. เตรียม features + target
features = ['lag1', 'lag2', 'lag3', 'lag4', 'lag5', 'volume']
# Xs = ...
# ys = ...

# 2. Split train/test ตาม year
# train_mask = ...
# test_mask  = ...

# 3. Fit 3 models


# 4. เปรียบเทียบ accuracy


# 5. สรุป:
print('\nสรุป: ...')


## Part 4: Full Comparison + ROC Curve

**Part นี้เราจะรวม classifiers ทั้งหมดมาเปรียบเทียบในที่เดียว** และ plot ROC Curve ซึ่งเป็นเครื่องมือสำคัญในการเปรียบเทียบ classifier โดยไม่ขึ้นกับ threshold ROC Curve แสดง trade-off ระหว่าง True Positive Rate (Recall) และ False Positive Rate ที่ threshold ต่างๆ classifier ที่ดีจะมี AUC (Area Under Curve) ใกล้ 1.0

In [ ]:
# ─── รวม classifiers ทั้งหมดในที่เดียว ─────────────────────────────────────────
# วัตถุประสงค์: เปรียบเทียบ 5 classifiers บน Default test set

# กำหนด classifiers ที่จะเปรียบเทียบ
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'LDA'               : LinearDiscriminantAnalysis(),
    'QDA'               : QuadraticDiscriminantAnalysis(),
    'Naive Bayes'       : GaussianNB(),
    'KNN (K=5)'         : KNeighborsClassifier(n_neighbors=5)
}

# Fit ทุก model และเก็บ metrics
results = []
for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    results.append({
        'Method'   : name,
        'Accuracy' : round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
        'Recall'   : round(recall_score(y_test, y_pred, zero_division=0), 4),
        'F1'       : round(f1_score(y_test, y_pred, zero_division=0), 4)
    })

results_df = pd.DataFrame(results)
print('Comparison Table — All Classifiers on Default Test Set:')
print(results_df.to_string(index=False))

In [ ]:
# ─── ROC Curve: Logistic Regression vs LDA vs QDA ─────────────────────────────
# วัตถุประสงค์: plot ROC Curve เพื่อเปรียบเทียบ classifier โดยไม่ขึ้นกับ threshold
# AUC ใกล้ 1.0 = ดี, AUC = 0.5 = random

fig, ax = plt.subplots(figsize=(8, 6))

# สี และ style สำหรับแต่ละ method
roc_methods = {
    'Logistic Regression': ('steelblue', '-'),
    'LDA'               : ('tomato', '--'),
    'QDA'               : ('green', '-.')
}

for name, (color, style) in roc_methods.items():
    # ดึง model ที่ fit ไว้แล้ว
    clf = classifiers[name]
    y_proba = clf.predict_proba(X_test)[:, 1]
    
    # คำนวณ ROC curve
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    
    ax.plot(fpr, tpr, color=color, linestyle=style, lw=2,
            label=f'{name} (AUC = {auc:.3f})')

# เส้น diagonal = random classifier
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC = 0.5)')

ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
ax.set_ylabel('True Positive Rate (Recall/Sensitivity)', fontsize=12)
ax.set_title('ROC Curve: Logistic vs LDA vs QDA on Default Dataset', fontsize=13)
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\nAUC ยิ่งใกล้ 1.0 ยิ่งดี — classifier ดีจะ hug มุมบนซ้ายของกราฟ')

### 🔰 TODO 4 (Hard): Full Classification Pipeline + Report

ถึงเวลาทดสอบทักษะรวมทั้งหมด คุณจะสร้าง **Breast Cancer Classification Pipeline** ซึ่งเป็น use case จริงในทางการแพทย์ — ทำนายว่า tumor เป็น malignant (ร้าย) หรือ benign (ไม่ร้าย) จาก cell features งานนี้ต้องการ **Recall สูง** (ไม่놓ง miss malignant) เพราะ FN มีราคาสูงมากในทางการแพทย์ Dataset ที่ใช้คือ Breast Cancer Wisconsin (มีใน sklearn)

**สิ่งที่ต้องทำ:**
1. โหลด `sklearn.datasets.load_breast_cancer()`
2. Split: test_size=0.25, random_state=42, stratify=y
3. **Standardize features**: ใช้ `StandardScaler` (สำคัญสำหรับ KNN!)
4. Fit ทุก 5 classifiers: LogisticRegression, LDA, QDA, GaussianNB, KNN(K=5)
5. สร้าง full comparison table (Accuracy, Precision, Recall, F1, AUC)
6. Plot ROC Curve ของทุก 5 classifiers
7. **Recommend**: ควรใช้ classifier ไหนสำหรับ breast cancer screening? เพราะอะไร?

**Hint**: `from sklearn.preprocessing import StandardScaler` → `scaler = StandardScaler()` → fit บน train เท่านั้น!

In [ ]:
# TODO 4: Full Classification Pipeline — Breast Cancer
# ─── TODO: เขียน code ที่นี่ ──────────────────────────────────────────────────
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler

# 1. โหลด dataset
bc = load_breast_cancer()
# Xbc = ...
# ybc = ...
# print(bc.DESCR[:500])  # ดู description

# 2. Split


# 3. Standardize (fit บน train, transform ทั้งคู่)


# 4+5. Fit ทุก classifiers + comparison table


# 6. ROC Curve


# 7. Recommendation:
print('\nRecommendation:')
print('...')


## Case Study: LDA สำหรับ Credit Risk Screening

**Scenario**  
ธนาคาร ABC ต้องการระบบ pre-screening ลูกค้าสินเชื่อก่อนส่งให้ทีม credit analyst ทบทวน เป้าหมายคือ filter ลูกค้าที่มีความเสี่ยง default สูงออกโดยอัตโนมัติ ทีม Data Science มีข้อมูลประวัติลูกค้า 10,000 รายที่มี label default/no-default

**Data**  
Default dataset: balance, income, student (3 features), n=10,000, imbalanced (3.3% default)

**Method — LDA เหมาะเพราะ:**  
- n=10,000 ไม่ใหญ่มาก LDA stable กว่า QDA  
- Classes well-separated (balance ต่างกันชัด)  
- Linear boundary อธิบายให้ regulator เข้าใจได้ง่าย

**Result**  
- LDA AUC = 0.948 (ดีกว่า QDA = 0.941 เล็กน้อย)  
- ปรับ threshold จาก 0.5 → 0.2: Recall ขึ้นจาก 0.62 → 0.84  
- จำนวน cases ที่ต้องทบทวน: เพิ่มขึ้น 15% แต่ catch default เพิ่ม 35%

**Insight**  
LDA ที่ปรับ threshold เป็นทางเลือกที่ดีสำหรับ credit screening — อธิบายได้ง่าย (linear boundary) AUC สูง และสามารถควบคุม recall ผ่าน threshold ได้ตาม business requirement

## สรุปสิ่งที่เรียนรู้

| Method | Assumption | Boundary | Python Class | เหมาะกับ |
|--------|-----------|----------|-------------|----------|
| Logistic Regression | ไม่มี distribution assumption | Linear | `LogisticRegression` | n ใหญ่, mild separation |
| LDA | Gaussian shared Σ | Linear | `LinearDiscriminantAnalysis` | n เล็ก, classes well-sep |
| QDA | Gaussian per-class Σₖ | Quadratic | `QuadraticDiscriminantAnalysis` | n ใหญ่, classes differ |
| Naive Bayes | Gaussian + independent | Flexible | `GaussianNB` | many features, text |
| KNN | ไม่มี | Non-linear | `KNeighborsClassifier` | n ใหญ่, complex boundary |

**Key takeaway**: ไม่มี classifier ที่ดีที่สุดในทุกสถานการณ์ — ต้องเลือกตาม data, n, class balance และ interpretability requirement

## คำถาม Reflection

1. **ถ้า features มี correlation สูง** (เช่น balance และ income มี r > 0.8) method ไหนระหว่าง LDA และ Naive Bayes จะได้รับผลกระทบมากกว่า? เพราะอะไร?

2. **ROC AUC กับ F1 Score** วัดสิ่งที่ต่างกัน ในสถานการณ์ใดที่คุณควรใช้ AUC แทน F1 ในการเปรียบเทียบ classifier?

3. **LDA และ PCA** ต่างกันอย่างไร? ทั้งคู่ใช้ covariance matrix — แต่เป้าหมายคืออะไร? (ใช้ความรู้จาก Week 4)